In [1]:
import pandas as pd
import numpy as np
import re
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Bidirectional
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
import pickle

In [2]:
df = pd.read_csv('kode_scraping.csv')  
df.head()

,text,rating,kategori
0,aplikasi canggih dan mudah dimengerti,5,Lainnya
1,Mudah dipakai,5,Lainnya
2,sangat membantu,5,Lainnya
3,"udah pakai kartu jakOne nya dari lama, ternyat...",5,UI/UX
4,kenapa jack one mobile sekarang baru daftar 1×...,3,Bug Teknis


In [3]:
stop_factory = StopWordRemoverFactory()
stop_remover = stop_factory.create_stop_word_remover()
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

def clean_text(text):
    if pd.isna(text):
        return ""
    text = re.sub(r'http\S+', ' ', text)                      
    text = re.sub(r'[^A-Za-zÀ-ÿ\s]', ' ', text)              
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()                 
    text = stop_remover.remove(text)                        
    return stemmer.stem(text) 

df['clean_text'] = df['text'].apply(clean_text)
df.head()

,text,rating,kategori,clean_text
0,aplikasi canggih dan mudah dimengerti,5,Lainnya,aplikasi canggih mudah erti
1,Mudah dipakai,5,Lainnya,mudah pakai
2,sangat membantu,5,Lainnya,sangat bantu
3,"udah pakai kartu jakOne nya dari lama, ternyat...",5,UI/UX,udah pakai kartu jakone nya lama nyata mobile ...
4,kenapa jack one mobile sekarang baru daftar 1×...,3,Bug Teknis,jack one mobile sekarang baru daftar jam passw...


In [4]:
def label_sentiment(rating):
    if rating <= 2:
        return 'negatif'
    elif rating == 3:
        return 'netral'
    else:
        return 'positif'

df['sentiment'] = df['rating'].apply(label_sentiment)

print("Distribusi sentimen:")
print(df['sentiment'].value_counts())

Distribusi sentimen:
sentiment
positif    8017
negatif    5193
netral      622
Name: count, dtype: int64


In [5]:
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['text'])
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
svm = SVC(kernel='linear')
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)

print(f"Akurasi Skema 1: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred))

Akurasi Skema 1: 90.50%
              precision    recall  f1-score   support

     negatif       0.84      0.96      0.89      1058
      netral       0.00      0.00      0.00       128
     positif       0.96      0.94      0.95      1581

    accuracy                           0.90      2767
   macro avg       0.60      0.63      0.61      2767
weighted avg       0.87      0.90      0.88      2767



C:\Users\shafira nabilazzahra\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\shafira nabilazzahra\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\shafira nabilazzahra\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parame

In [6]:
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['text']).toarray()
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print(f"Akurasi Skema 2: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred))

Akurasi Skema 2: 89.23%
              precision    recall  f1-score   support

     negatif       0.82      0.95      0.88      1058
      netral       0.00      0.00      0.00       128
     positif       0.96      0.93      0.94      1581

    accuracy                           0.89      2767
   macro avg       0.59      0.62      0.61      2767
weighted avg       0.86      0.89      0.87      2767



In [16]:
max_words = 5000
maxlen = 100

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['text'])
X_seq = tokenizer.texts_to_sequences(df['text'])
X_pad = pad_sequences(X_seq, maxlen=maxlen)

y = pd.Categorical(df['sentiment']).codes
y_cat = to_categorical(y)

X_train, X_test, y_train, y_test = train_test_split(X_pad, y_cat, test_size=0.2, random_state=42)

model = Sequential([
    Embedding(max_words, 128, input_length=maxlen),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history = model.fit(X_train, y_train, validation_data=(X_test, y_test),
                    epochs=5, batch_size=64, verbose=1)

loss, acc = model.evaluate(X_test, y_test)
print(f"Akurasi Skema 3 (LSTM): {acc * 100:.2f}%")

C:\Users\shafira nabilazzahra\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
173/173 ━━━━━━━━━━━━━━━━━━━━ 45s 198ms/step - accuracy: 0.8107 - loss: 0.5143 - val_accuracy: 0.9013 - val_loss: 0.3139
Epoch 2/5
173/173 ━━━━━━━━━━━━━━━━━━━━ 31s 177ms/step - accuracy: 0.9032 - loss: 0.3117 - val_accuracy: 0.9031 - val_loss: 0.2949
Epoch 3/5
173/173 ━━━━━━━━━━━━━━━━━━━━ 29s 168ms/step - accuracy: 0.9141 - loss: 0.2546 - val_accuracy: 0.8999 - val_loss: 0.3157
Epoch 4/5
173/173 ━━━━━━━━━━━━━━━━━━━━ 29s 166ms/step - accuracy: 0.9197 - loss: 0.2254 - val_accuracy: 0.8876 - val_loss: 0.3272
Epoch 5/5
173/173 ━━━━━━━━━━━━━━━━━━━━ 29s 167ms/step - accuracy: 0.9301 - loss: 0.2006 - val_accuracy: 0.8865 - val_loss: 0.3668
87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - accuracy: 0.8865 - loss: 0.3668
Akurasi Skema 3 (LSTM): 88.65%


In [13]:
y_pred1 = svm.predict(X_test)

teks_asli = df.loc[y_test.index, 'text']

df_pred1 = pd.DataFrame({
    "Teks": teks_asli[:10],
    "Actual": y_test[:10].values,
    "Prediksi": y_pred[:10]
})

print(df_pred1)

                                                    Teks   Actual Prediksi
5828   Aplikasi yang bagus⭐⭐⭐⭐⭐ sistem nya sangat mud...  positif  positif
6729   apk nya kurang banget soal sistem nya, ga dlu ...  negatif  negatif
12126  Merchant-merchanya cukup lengkap, mulai dari m...  positif  positif
2974   kecewa banget, knapa sih transaksi tf ke bank ...  negatif  negatif
2308   kenapa tidak bisa transfer ke bank lain? sanga...  negatif  negatif
360    idiottt. aplikasi ga guna. nge bug terus. pake...  negatif  negatif
4401                     Sangat bagus untuk bertransaksi  positif  positif
1586                                     aplikasi sampah  negatif  negatif
9037   Aplikasi yng bagus mempermudah kita untuk mela...  positif  positif
5309   Pendaftaran cepat. Dan pasti di setujui. Bagi ...  positif  positif


In [14]:
y_pred2 = rf.predict(X_test)

df_pred2 = pd.DataFrame({
    "Teks": df['text'].iloc[y_test.index][:10],
    "Actual": y_test[:10].values,
    "Prediksi": y_pred[:10]
})

print(df_pred2)


                                                    Teks   Actual Prediksi
5828   Aplikasi yang bagus⭐⭐⭐⭐⭐ sistem nya sangat mud...  positif  positif
6729   apk nya kurang banget soal sistem nya, ga dlu ...  negatif  negatif
12126  Merchant-merchanya cukup lengkap, mulai dari m...  positif  positif
2974   kecewa banget, knapa sih transaksi tf ke bank ...  negatif  negatif
2308   kenapa tidak bisa transfer ke bank lain? sanga...  negatif  negatif
360    idiottt. aplikasi ga guna. nge bug terus. pake...  negatif  negatif
4401                     Sangat bagus untuk bertransaksi  positif  positif
1586                                     aplikasi sampah  negatif  negatif
9037   Aplikasi yng bagus mempermudah kita untuk mela...  positif  positif
5309   Pendaftaran cepat. Dan pasti di setujui. Bagi ...  positif  positif


In [24]:
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

label_map = {0: "negatif", 1: "netral", 2: "positif"}

# Konversi hasil prediksi & aktual ke label teks
y_pred_labels = [label_map[i] for i in y_pred]
y_true_labels = [label_map[i] for i in y_true]

# --- Buat DataFrame hasil prediksi ---
# Ambil teks asli yang sesuai dengan X_test
# Gunakan index dari train_test_split (tanpa shuffle, random_state tetap 42)
X_text = np.array(df['text'])
_, X_test_text = train_test_split(X_text, test_size=0.2, random_state=42)

df_pred3 = pd.DataFrame({
    "Teks": X_test_text[:10],
    "Actual": y_true_labels[:10],
    "Prediksi": y_pred_labels[:10]
})

print("\nContoh hasil prediksi:")
print(df_pred3)

87/87 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step

Contoh hasil prediksi:
                                                Teks   Actual Prediksi
0  Aplikasi yang bagus⭐⭐⭐⭐⭐ sistem nya sangat mud...  positif  positif
1  apk nya kurang banget soal sistem nya, ga dlu ...  negatif  negatif
2  Merchant-merchanya cukup lengkap, mulai dari m...  positif  positif
3  kecewa banget, knapa sih transaksi tf ke bank ...  negatif  negatif
4  kenapa tidak bisa transfer ke bank lain? sanga...  negatif  negatif
5  idiottt. aplikasi ga guna. nge bug terus. pake...  negatif  negatif
6                    Sangat bagus untuk bertransaksi  positif  positif
7                                    aplikasi sampah  negatif  negatif
8  Aplikasi yng bagus mempermudah kita untuk mela...  positif  positif
9  Pendaftaran cepat. Dan pasti di setujui. Bagi ...  positif  positif
